# Outlines
* Required Installation
* XOR Arbiter PUF
* QR-PUF
* BB84 Qbit
* Implementation
    * A-PUF Implementation
    * Qiskit Encoding & Circuit Measurement
    * ML-Attack (Logistic Regression)

## Required Installation

Run the cell below once to install the required packages. (Drop `--break-system-packages` if you're running inside a virtual environment — it's only needed on systems with an "externally managed" Python installation, e.g. recent Debian/Ubuntu.)

In [3]:
%pip install pypuf==3.2.1 --no-deps --break-system-packages
%pip install tensorflow --break-system-packages
%pip install "numpy<2.0" --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pypuf 3.2.1 requires tensorflow~=2.4.0, but you have tensorflow 2.21.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached nu

# XOR Arbiter PUF 

**Definition:** XOR Arbiter PUF combines $k$ **independent** classical APUF, each fed with the **same challenge**, by **XOR-ing** their outputs together.

## Additive Delay Model

Each component ($l = 1, \dots, k$) follows the standard APUF additive model on its own:

$$\Delta^{(l)} = \mathbf{w}^{(l)T} \boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}$$

- $\boldsymbol{\phi}(\mathbf{c})$: the parity vector, **shared across all components** (they all receive the same challenge)
- $\mathbf{w}^{(l)}$: each component's **own** chip-specific weight vector (different physical MUX chains → different manufacturing variation)

Each component's own response:
$$r^{(l)} = \text{sign}(-\Delta^{(l)})$$

The final response is their XOR:
$$r = r^{(1)} \oplus r^{(2)} \oplus \dots \oplus r^{(k)}$$

Since XOR becomes a **product** in the bipolar $\{-1,+1\}$ representation:

$$r' = \prod_{l=1}^{k} \text{sign}\left(-\Delta^{(l)}\right) = \prod_{l=1}^{k} \text{sign}\left(\mathbf{w}^{(l)T}\boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}\right)$$

## What Changes, in Short

| | Classical APUF | XOR APUF |
|---|---|---|
| $\Delta$ | A single linear sum: $\mathbf{w}^T\phi+w_0$ | **Product of the signs** of $k$ separate linear sums |
| Linearity | Fully linear in $\phi(\mathbf{c})$ | Linear within each component, but the combination is non-linear |
| ML resistance | Weak (broken by simple logistic regression) | Increases with $k$ (required CRP count grows exponentially) |
| Reliability | High | Decreases as $k$ grows (noise accumulates through the XOR) |

**In essence:** each component still belongs to the "additive delay model" family — internally it's just a sum $\mathbf{w}^{(l)T}\phi+w_0^{(l)}$. What makes the XOR-PUF special is combining these $k$ linear components with an **XOR (multiplicative) operator**, which turns an otherwise linear structure into a function that is roughly degree-$k$ non-linear in $\phi(\mathbf{c})$ — exactly the mechanism that boosts security while trading off reliability.

## QR-PUF
As we saw in the previous lab, classical Arbiter PUFs (A-PUFs), while physically unclonable, are **mathematically modelable**. Given enough challenge-response pairs (CRPs), an attacker (Eve) can use a simple ML model like logistic regression to clone the PUF's behavior with high accuracy (~98-99%). This is fundamentally because the PUF's additive delay model is **linearly separable** under the φ-parity transformation.

This vulnerability raises the question: *What if the responses are not given to the attacker directly as observable classical bits?*

The Quantum-Readout PUF (QR-PUF), proposed by Škorić, is an approach where **the physical PUF structure remains classical**, but the communication between the PUF and the verifier - i.e., how the challenge and response are carried — happens through **quantum states**.

The **HPUF (Hybrid Locked PUF)** construction we'll use in this lab is a concrete instance of this idea:

$$
\text{Classical PUF output} \;\xrightarrow{\text{BB84 encoding}}\; \text{Non-orthogonal quantum state}
$$

wo fundamental properties of quantum mechanics come into play here:

1. **No-cloning theorem:** A perfect copy of an unknown quantum state cannot be produced. The attacker cannot "copy" the PUF's quantum response to both forward it to the verifier and use it for their own analysis.

2. **Indistinguishability of non-orthogonal states:** If the response is encoded by randomly choosing between two different bases (Z and X), the attacker cannot determine with certainty, **from a single copy**, which basis was used. If they measure in the wrong basis, the result they obtain is a **completely random bit with no relation to the original value**.

## BB84 Qubit Encoding

BB84 is the protocol proposed by Bennett and Brassard in 1984, forming the foundation of quantum key distribution (QKD). In this lab, we are not using BB84 as a key-distribution protocol, but rather as a **single information-encoding scheme**: to convert the classical bits produced by the PUF into unclonable quantum states.

BB84 encoding uses two different measurement bases:

| Basis | Symbol | 0 → State | 1 → State |
|---|---|---|---|
| Rectilinear (Z basis) | $\{\lvert 0\rangle, \lvert 1\rangle\}$ | $\lvert 0\rangle$ | $\lvert 1\rangle$ |
| Diagonal (X basis) | $\{\lvert +\rangle, \lvert -\rangle\}$ | $\lvert +\rangle = \frac{\lvert 0\rangle + \lvert 1\rangle}{\sqrt{2}}$ | $\lvert -\rangle = \frac{\lvert 0\rangle - \lvert 1\rangle}{\sqrt{2}}$ |

These four states — $\lvert 0\rangle, \lvert 1\rangle, \lvert +\rangle, \lvert -\rangle$ — are **non-orthogonal** to each other (between the Z and X bases). This is the cornerstone of the protocol's security.


Each BB84 qubit carries **two classical bits**:

- **Basis bit:** determines which basis is used (0 → Z basis, 1 → X basis)
- **Value bit:** determines which state within that basis is encoded (0 or 1)

In this lab, we convert the response bit-pairs (basis_bit, value_bit) coming from the A-PUF into a qubit as follows:

$$
\lvert \psi \rangle =
\begin{cases}
\lvert 0 \rangle & \text{basis}=0,\ \text{value}=0 \\
\lvert 1 \rangle & \text{basis}=0,\ \text{value}=1 \\
\lvert + \rangle & \text{basis}=1,\ \text{value}=0 \\
\lvert - \rangle & \text{basis}=1,\ \text{value}=1
\end{cases}
$$

When the qubit reaches the receiver (or an eavesdropping attacker), which basis it was prepared in is **physically invisible** — the qubit itself does not reveal this. The receiver must choose a basis and measure:

- **If they choose the correct basis** (e.g., measuring in the X basis a qubit that was actually prepared in the X basis): the value bit is read back **with certainty**.
- **If they choose the wrong basis** (e.g., measuring in the Z basis a qubit prepared in the X basis): by the fundamental postulates of quantum mechanics, the measurement outcome is **50% 0, 50% 1** — carrying **no information whatsoever** about the original value bit.

This behavior will be numerically verified later, when we simulate Eve's measurement strategy: since her basis guess is random (50% correct), and the outcome under a wrong-basis measurement is fully random, the labels Eve ends up with form a noisy dataset that deviates from the true PUF response (bit-flip) on average **25% of the time**.


### XOR-APUF Implementation

Following codes presents A-PUF implementation(see LAB 1):

In [19]:
import numpy as np
from pypuf.simulation import XORArbiterPUF
from pypuf.io import random_inputs

n = 64
k = 4
puf = XORArbiterPUF(n=n, k=k, seed=1, noisiness=0.0)

train_N = 100000
test_N  = 10000
train_challenges = random_inputs(n=n, N=train_N, seed=2)
train_responses = puf.eval(train_challenges)    
test_challenges = random_inputs(n=n, N=test_N, seed=2)
test_responses = puf.eval(test_challenges)  

### ML Attack (Logistic Regression) for XOR-APUF

Similar steps are applied with Lab 1.

In [20]:
import matplotlib.pyplot as plt
import tensorflow as tf
from pypuf.simulation.base import LTFArray
from pypuf.io import random_inputs, ChallengeResponseSet
from pypuf.attack import LRAttack2021
import pypuf.attack

train_N = 50000
crps = ChallengeResponseSet(train_challenges[:train_N], train_responses[:train_N])
attack = LRAttack2021(crps, seed=100, k=k, bs=1000, lr=0.001, epochs=100, stop_validation_accuracy=1.0)
model = attack.fit()

predicted = model.eval(test_challenges)
accuracy = np.mean(
predicted.reshape(-1) == test_responses.reshape(-1)
)
print('Attack accuracy:', accuracy)

Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4987 - loss: 0.6932 - val_accuracy: 0.5300 - val_loss: 0.6930
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5085 - loss: 0.6929 - val_accuracy: 0.5080 - val_loss: 0.6931
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5087 - loss: 0.6926 - val_accuracy: 0.5060 - val_loss: 0.6932
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5104 - loss: 0.6923 - val_accuracy: 0.4960 - val_loss: 0.6933
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5162 - loss: 0.6918 - val_accuracy: 0.5300 - val_loss: 0.6935
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5170 - loss: 0.6913 - val_accuracy: 0.5260 - val_loss: 0.6937
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5197 - loss: 0.6908 - val_accuracy: 0.5200 - val_loss: 0.6939
Epoch 8/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5217 - loss: 0.6903 - val_accuracy: 0.5220 - v

**WARNING:** If you encounter an error like; 


"Traceback (most recent call last):
  File "/home/seuz/QPUF Tutorial/deneme.py", line 27, in <module>
    model = attack.fit()
  File "/home/seuz/.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 145, in fit
    self._history = model.fit(
  File "/home/seuz/.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/home/seuz/.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 89, in accuracy
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)
TypeError: Expected int8, but got 0.5 of type 'float'."

run following cell and train again.

In [5]:
def keras_to_pypuf_fixed(self, keras_model):
    n, k = self.crps.challenges.shape[1], self.k
    weights = np.zeros(shape=(k, n))
    bias = np.zeros(shape=(k,))
    for l in range(k):
        layer_weights = keras_model.layers[l + 1].get_weights()
        weights[l] = layer_weights[0][:, 0]
        bias[l] = layer_weights[1][0]
    return LTFArray(weight_array=weights, bias=bias, transform=XORArbiterPUF.transform_atf)
pypuf.attack.LRAttack2021.keras_to_pypuf = keras_to_pypuf_fixed

@staticmethod
def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.losses.binary_crossentropy(.5 - .5 * y_true, .5 - .5 * y_pred)

@staticmethod
def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)

pypuf.attack.LRAttack2021.loss = loss
pypuf.attack.LRAttack2021.accuracy = accuracy

### BB84-Encoded PUF Responses
For each challenge $c_i$, compute $b_i = f_b(c_i)$ **(basis)** and $\theta_i = f_\theta(c_i)$ **(value)**, then conceptually return the BB84 state $|\psi_{(b_i,\theta_i)}\rangle$ according to the mapping.

### 2.3 Threat Model

The adversary receives:

$$
D_Q = \{(c_i, |\psi_{(b_i,\theta_i)}\rangle)\}_{i=1}^{q}
$$

Assume that the adversary:
- knows the PUF architecture and the BB84 encoding rule;
- does not know the hidden physical parameters of $f_b$ or $f_\theta$;
- receives randomly selected challenges and one corresponding quantum response state per challenge;
- has arbitrary quantum measurement capability, but only one copy of each state;
- cannot obtain repeated copies of the same training response state; and
- wants to build a classical ML model that predicts the underlying target response bit $b$ for unseen challenges.

> **Critical assumption:** The one-copy restriction is essential. With repeated copies of the same BB84 state, the adversary could improve state discrimination, so a fixed 14.64% effective label-error model would no longer represent the attack.

### 2.4 From One-Copy Quantum Measurement

If $b=0$, the emitted state is equally likely to be $|0\rangle$ or $|+\rangle$. If $b=1$, it is equally likely to be $|1\rangle$ or $|-\rangle$. The adversary therefore distinguishes the two mixed states:

$$
\rho_0 = \frac{1}{2}\big(|0\rangle\langle 0| + |+\rangle\langle +|\big), \qquad
\rho_1 = \frac{1}{2}\big(|1\rangle\langle 1| + |-\rangle\langle -|\big)
$$

For unbiased bits, optimal single-copy discrimination gives:

$$
p_{guess} = \frac{1}{2} + \frac{1}{2\sqrt{2}} \approx 0.853553, \qquad
p_{error} = 1 - p_{guess} = \frac{1}{2} - \frac{1}{2\sqrt{2}} \approx 0.146447
$$

**Interpretation:** The ~14.64% is not physical noise introduced by the PUF or by BB84 preparation. It is the effective classical label-error rate after the optimal adversarial measurement of one copy of the encoded state.

In this lab, optimal-measurement stages can be replaced by their analytically known classical statistics:

$$
b \rightarrow |\psi\rangle \rightarrow \text{optimal measurement} \rightarrow \tilde{b}
$$

$$
\Pr[\tilde{b} = b] = 0.853553, \qquad \Pr[\tilde{b} \neq b] = 0.146447
$$

We generate HPUF responses by equation given by above.


In [21]:
clean_responses = train_responses.copy()

p_guess = 0.5 + 1/(2*np.sqrt(2))
p_error = 1 - p_guess
print('p_guess =', p_guess)
print('p_error =', p_error)

rng = np.random.default_rng(12345)
measured_responses = clean_responses.copy()
flip = rng.random(len(measured_responses)) < p_error
measured_responses[flip] *= -1

observed_error = np.mean(
    measured_responses.reshape(-1) != clean_responses.reshape(-1)
)
print('Observed extraction error:', observed_error)

p_guess = 0.8535533905932737
p_error = 0.14644660940672627
Observed extraction error: 0.14624


For a large training database, the observed extraction error should be close to 0.1464.

### Modeling Attack (Logistic Regression Attack) for HPUF

In [22]:
train_N = 50000
quantum_crps = ChallengeResponseSet(
    train_challenges[:train_N],
    measured_responses[:train_N]
)
attack_q = LRAttack2021(
    quantum_crps,
    seed=100,
    k=k,
    bs=1000,
    lr=0.001,
    epochs=100,
    stop_validation_accuracy=1.0
)
model_q = attack_q.fit()

predicted_q = model_q.eval(test_challenges)
accuracy_q = np.mean(
    predicted_q.reshape(-1) == test_responses.reshape(-1)
)
print('BB84-interface attack accuracy:', accuracy_q)

Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4954 - loss: 0.6933 - val_accuracy: 0.4700 - val_loss: 0.6942
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5024 - loss: 0.6930 - val_accuracy: 0.4980 - val_loss: 0.6942
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5053 - loss: 0.6927 - val_accuracy: 0.4960 - val_loss: 0.6945
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5102 - loss: 0.6925 - val_accuracy: 0.4900 - val_loss: 0.6949
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5118 - loss: 0.6921 - val_accuracy: 0.4920 - val_loss: 0.6956
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5165 - loss: 0.6917 - val_accuracy: 0.4720 - val_loss: 0.6964
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5182 - loss: 0.6913 - val_accuracy: 0.4680 - val_loss: 0.6972
Epoch 8/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5202 - loss: 0.6909 - val_accuracy: 0.4640 - v